<div style="font-family:'Times New Roman'; text-align:center;">


### * Implementasi Analisis Data Seismik *

<div style="text-align:center;">
  <img src="https://www.unhas.ac.id/wp-content/uploads/2022/09/Logo-Resmi-Unhas-1.png" alt="Logo Unhas" style="width:280px;">
</div>

### *DISUSUN OLEH :*


<div style="font-size: 20px; line-height: 1.2;">
  <div>AURAH PUTRI NATASYA (H061241020)</div>
  
</div>

### *PROGRAM STUDI GEOFISIKA*
### *FAKULTAS MATEMATIKA DAN ILMU PENGETAHUAN ALAM*
#### *UNIVERSITAS HASANUDDIN*
#### *MAKASSAR*
#### *2026*

<hr>

</div>



## Informasi Recipe

Recipe ID: R-ANA-001 (adaptasi untuk data seismik)  
Chapter: Data Cleaning, QA/QC, and Exploration  
Bidang: Geofisika — Seismologi  
Data: Gelombang seismik sintetis dari tiga stasiun  
Durasi simulasi: 60 detik  
Catatan: Dataset pada notebook ini **merupakan data sintetis untuk latihan**, bukan rekaman gempa bumi nyata atau data dari stasiun seismik tertentu.

## 1. Tujuan

Notebook ini bertujuan melakukan eksplorasi awal terhadap data seismik, meliputi pemeriksaan kualitas data, visualisasi bentuk gelombang, perhitungan amplitudo, nilai RMS, dan estimasi frekuensi dominan. Hasilnya digunakan untuk mengenali karakteristik sinyal sebelum dilakukan analisis seismik lanjutan.

## 2. Data yang digunakan

Data terdiri atas tiga stasiun sintetis dengan laju sampling 100 sampel per detik selama 60 detik. Sinyal dibuat dari kombinasi gelombang sinus, gangguan acak, dan satu kejadian impuls sederhana untuk meniru bentuk sinyal seismik secara ilustratif.

## 3. Metode

1. Membuat data gelombang seismik sintetis.
2. Memeriksa nilai kosong, nilai tak hingga, dan duplikasi waktu.
3. Menghitung amplitudo maksimum absolut, RMS, dan frekuensi dominan.
4. Membuat grafik gelombang pada domain waktu dan spektrum frekuensi.
5. Menyimpan data dan ringkasan hasil ke folder keluaran.

**Penting:** hasil analisis hanya menggambarkan sinyal sintetis yang dibuat oleh kode. Hasil tidak boleh ditafsirkan sebagai parameter gempa nyata.

In [ ]:
# ============================================================
# RECIPE R-ANA-001: EXPLORATORY SEISMIC DATA SUMMARY
# Implementasi dengan data seismik sintetis
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# PENGATURAN
FOLDER_OUTPUT = Path("hasil_seismik_R_ANA_001")
FOLDER_OUTPUT.mkdir(parents=True, exist_ok=True)

FREKUENSI_SAMPLING = 100  # sampel per detik (Hz)
DURASI_DETIK = 60
rng = np.random.default_rng(42)

# 1. MEMBUAT DATA SEISMIK SINTETIS
# Data dibuat untuk latihan, bukan rekaman lapangan.
waktu = np.arange(0, DURASI_DETIK, 1 / FREKUENSI_SAMPLING)

def buat_sinyal(f1, f2, amplitudo, waktu_impuls, seed_noise):
    """Membuat sinyal ilustratif dengan dua frekuensi dan satu impuls."""
    noise_rng = np.random.default_rng(seed_noise)
    sinyal = (
        amplitudo * np.sin(2 * np.pi * f1 * waktu)
        + 0.35 * amplitudo * np.sin(2 * np.pi * f2 * waktu)
        + 0.12 * amplitudo * noise_rng.normal(size=len(waktu))
    )
    # Impuls teredam sederhana untuk meniru kedatangan gelombang.
    indeks = waktu >= waktu_impuls
    tau = waktu[indeks] - waktu_impuls
    sinyal[indeks] += (
        1.8 * amplitudo * np.exp(-tau / 1.2)
        * np.sin(2 * np.pi * 8 * tau)
    )
    return sinyal

data = pd.DataFrame({
    "waktu_detik": waktu,
    "STA01": buat_sinyal(5, 12, 1.0, 12.0, 1),
    "STA02": buat_sinyal(7, 18, 0.8, 15.0, 2),
    "STA03": buat_sinyal(4, 10, 1.2, 18.0, 3),
})

display(data.head())
print(f"Jumlah sampel: {len(data):,}")
print(f"Frekuensi sampling: {FREKUENSI_SAMPLING} Hz")
print(f"Durasi data: {DURASI_DETIK} detik")

## 4. Pemeriksaan Kualitas Data (QA/QC)

Pemeriksaan dilakukan untuk memastikan kolom waktu dan sinyal dapat dianalisis. Pemeriksaan meliputi nilai kosong, nilai tak hingga, dan waktu duplikat.

In [ ]:
# 2. PEMERIKSAAN KUALITAS DATA
kolom_sinyal = ["STA01", "STA02", "STA03"]

jumlah_missing = data[kolom_sinyal].isna().sum()
jumlah_infinite = pd.Series({
    kolom: int(np.isinf(data[kolom].to_numpy()).sum())
    for kolom in kolom_sinyal
})
jumlah_waktu_duplikat = int(data["waktu_detik"].duplicated().sum())

laporan_qc = pd.DataFrame({
    "missing": jumlah_missing,
    "infinite": jumlah_infinite,
    "jumlah_sampel": data[kolom_sinyal].count()
})
display(laporan_qc)

print("Jumlah waktu duplikat:", jumlah_waktu_duplikat)
print("Waktu monoton naik:", bool(data["waktu_detik"].is_monotonic_increasing))

if jumlah_missing.sum() == 0 and jumlah_infinite.sum() == 0 and jumlah_waktu_duplikat == 0:
    print("QA/QC dasar: tidak ditemukan nilai kosong, tak hingga, atau waktu duplikat.")
else:
    print("QA/QC dasar: terdapat masalah yang perlu diperiksa.")

## 5. Ringkasan Parameter Sinyal

Amplitudo maksimum absolut menunjukkan nilai puncak terbesar tanpa membedakan tanda positif atau negatif. RMS (*root mean square*) menggambarkan besar sinyal rata-rata secara kuadratik. Frekuensi dominan diperkirakan dari frekuensi dengan amplitudo spektrum terbesar, dengan komponen frekuensi nol dikeluarkan.

In [ ]:
# 3. MENGHITUNG PARAMETER DASAR
ringkasan = []

for kolom in kolom_sinyal:
    sinyal = data[kolom].to_numpy()
    spektrum = np.abs(np.fft.rfft(sinyal - np.mean(sinyal)))
    frekuensi = np.fft.rfftfreq(
        len(sinyal), d=1 / FREKUENSI_SAMPLING
    )

    # Mengabaikan frekuensi nol agar komponen DC tidak menjadi dominan.
    indeks_dominan = 1 + np.argmax(spektrum[1:])
    frekuensi_dominan = frekuensi[indeks_dominan]

    ringkasan.append({
        "stasiun": kolom,
        "amplitudo_maks_abs": np.max(np.abs(sinyal)),
        "amplitudo_min": np.min(sinyal),
        "amplitudo_maks": np.max(sinyal),
        "RMS": np.sqrt(np.mean(sinyal ** 2)),
        "frekuensi_dominan_Hz": frekuensi_dominan,
    })

df_ringkasan = pd.DataFrame(ringkasan).set_index("stasiun")
display(df_ringkasan.round(4))

## 6. Visualisasi Gelombang Seismik

Grafik berikut memperlihatkan perubahan amplitudo terhadap waktu pada masing-masing stasiun sintetis. Perbedaan bentuk sinyal terjadi karena setiap stasiun dibuat dengan frekuensi, amplitudo, dan waktu impuls yang berbeda.

In [ ]:
# 4. GRAFIK DOMAIN WAKTU
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)

for ax, kolom in zip(axes, kolom_sinyal):
    ax.plot(data["waktu_detik"], data[kolom], linewidth=0.7)
    ax.set_ylabel("Amplitudo")
    ax.set_title(f"Bentuk gelombang {kolom}")
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("Waktu (detik)")
fig.suptitle("Data Seismik Sintetis — Domain Waktu", y=1.01)
fig.tight_layout()
fig.savefig(FOLDER_OUTPUT / "gelombang_seismik.png", dpi=200, bbox_inches="tight")
plt.show()

## 7. Analisis Spektrum Frekuensi

Spektrum frekuensi membantu melihat komponen frekuensi yang menyusun sinyal. Puncak spektrum pada data sintetis berkaitan dengan frekuensi yang digunakan saat membentuk sinyal, tetapi dapat dipengaruhi oleh noise dan impuls.

In [ ]:
# 5. GRAFIK SPEKTRUM FREKUENSI
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)

for ax, kolom in zip(axes, kolom_sinyal):
    sinyal = data[kolom].to_numpy()
    spektrum = np.abs(np.fft.rfft(sinyal - np.mean(sinyal)))
    frekuensi = np.fft.rfftfreq(
        len(sinyal), d=1 / FREKUENSI_SAMPLING
    )
    ax.plot(frekuensi, spektrum, linewidth=0.8)
    ax.set_xlim(0, 30)
    ax.set_ylabel("Magnitudo")
    ax.set_title(f"Spektrum {kolom}")
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("Frekuensi (Hz)")
fig.suptitle("Spektrum Frekuensi Data Seismik Sintetis", y=1.01)
fig.tight_layout()
fig.savefig(FOLDER_OUTPUT / "spektrum_frekuensi.png", dpi=200, bbox_inches="tight")
plt.show()

## 8. Menyimpan Hasil

Data sintetis, tabel ringkasan parameter, dan laporan QA/QC disimpan sebagai file CSV. Grafik disimpan sebagai PNG di folder `hasil_seismik_R_ANA_001`.

In [ ]:
# 6. MENYIMPAN DATA DAN HASIL
data.to_csv(FOLDER_OUTPUT / "data_seismik_sintetis.csv", index=False)
df_ringkasan.to_csv(FOLDER_OUTPUT / "ringkasan_parameter_seismik.csv")
laporan_qc.to_csv(FOLDER_OUTPUT / "laporan_qc.csv")

print("File hasil disimpan di:", FOLDER_OUTPUT.resolve())
for file_hasil in sorted(FOLDER_OUTPUT.iterdir()):
    print("-", file_hasil.name)

## Kesimpulan

Berdasarkan analisis eksploratif terhadap data seismik sintetis, setiap stasiun menunjukkan bentuk gelombang dengan karakteristik amplitudo dan frekuensi yang berbeda. Parameter amplitudo maksimum absolut dan RMS memberikan gambaran tingkat amplitudo sinyal, sedangkan spektrum frekuensi digunakan untuk memperkirakan frekuensi dominan.

Pemeriksaan QA/QC dasar dilakukan untuk mengecek nilai kosong, nilai tak hingga, serta duplikasi waktu. Grafik domain waktu dan spektrum frekuensi membantu meninjau karakteristik sinyal sebelum analisis lebih lanjut.

Notebook ini merupakan contoh alur kerja eksplorasi data seismik menggunakan Python. Karena data dibuat secara sintetis, hasilnya tidak dapat digunakan untuk menyimpulkan lokasi sumber gempa, magnitudo, kedalaman, atau kondisi geologi sebenarnya. Untuk analisis lapangan, data perlu diganti dengan rekaman seismik asli beserta metadata stasiun dan informasi sampling yang benar.